# 👁️ Pupil-Limbus Segmentation Model Training on Google Colab 🚀

Welcome to the Google Colab training notebook for the Pupil-Limbus eye tracker!  
By using Google Colab's high-performance cloud GPUs (such as the NVIDIA T4, L4, or A100), you can train your segmentation models **up to 20x faster** than on a standard laptop CPU or low-end GPU.

### 📋 How to Prepare and Run Curation & Training:
1. **Zip your clinical data locally**: Compress your local `clinical_data` directory into a file named `clinical_data.zip`.
2. **Zip your project code locally**: Compress the code files (excluding `.venv`, `clinical_data`, and `models` to keep it small) into a file named `project_code.zip`.
3. **Upload both files to Google Colab**: Use the left sidebar folder icon, or connect Google Drive (recommended for larger datasets) to copy them over.

In [ ]:
# @title 1. Verify GPU Acceleration 🏎️
# Ensure a high-performance GPU is allocated to your Colab session.
import torch
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    device_prop = torch.cuda.get_device_properties(0)
    print(f"✅ GPU active: {device_name}")
    print(f"Total GPU memory: {device_prop.total_memory / (1024**3):.2f} GB")
else:
    print("❌ GPU is NOT active!")
    print("Please go to Menu: Runtime -> Change runtime type -> Select GPU (T4 GPU is free) -> Save, then re-run this cell.")

## 2. Connect Your Datasets 📦

Choose one of the options below to connect your files to this notebook:

In [ ]:
# @title Option A: Mount Google Drive (Recommended) 📁
# Upload 'clinical_data.zip' and 'project_code.zip' to the root folder of your Google Drive first,
# then run this cell to mount Drive and copy the files to Colab's fast local storage.
from google.colab import drive
import shutil
from pathlib import Path

try:
    drive.mount('/content/drive')
    
    # Copy clinical data
    drive_data_zip = Path('/content/drive/MyDrive/clinical_data.zip')
    if drive_data_zip.exists():
        shutil.copy2(drive_data_zip, '/content/clinical_data.zip')
        print("✅ Copied clinical_data.zip from Drive to Colab local storage!")
    else:
        print("ℹ️ 'clinical_data.zip' not found in Drive. You can upload it manually via the sidebar.")

    # Copy project code
    drive_code_zip = Path('/content/drive/MyDrive/project_code.zip')
    if drive_code_zip.exists():
        shutil.copy2(drive_code_zip, '/content/project_code.zip')
        print("✅ Copied project_code.zip from Drive to Colab local storage!")
    else:
        print("ℹ️ 'project_code.zip' not found in Drive. You can upload it manually via the sidebar.")
except Exception as e:
    print(f"Google Drive mount skipped/failed: {e}")
    print("Please upload your ZIP files directly using the left folder icon instead.")

In [ ]:
# @title Option B: Extract ZIP Files 🔓
# Unzips training data and project source code to build the workspace structure.
import zipfile
import os
import shutil
from pathlib import Path

code_zip = Path("/content/project_code.zip")
data_zip = Path("/content/clinical_data.zip")

# Clean up previous extractions
if Path("/content/workspace").exists():
    shutil.rmtree("/content/workspace")

if code_zip.exists():
    print("Extracting project code...")
    with zipfile.ZipFile(code_zip, 'r') as zip_ref:
        zip_ref.extractall("/content/workspace")
    print("✅ Project code extracted!")
else:
    print("❌ 'project_code.zip' not found! Please upload it via the left panel.")

if data_zip.exists():
    print("Extracting clinical data...")
    # Extract clinical data directly inside the workspace folder
    with zipfile.ZipFile(data_zip, 'r') as zip_ref:
        zip_ref.extractall("/content/workspace")
    print("✅ Clinical data unzipped inside workspace!")
else:
    print("❌ 'clinical_data.zip' not found! Please upload it via the left panel.")

## 3. Install Requirements & Setup Environment 🛠️

In [ ]:
# @title Install Core Libraries & Dependencies 🚀
# Installs segmentation models and advanced image augmenters.
%cd /content/workspace

# Install requirements
!pip install -q segmentation-models-pytorch albumentations==1.4.3 onnxruntime-gpu

# Verify imports
import numpy as np
import cv2
import albumentations as A
import segmentation_models_pytorch as smp
print("\n✅ All packages imported and ready!")

## 4. Run Training Pipeline 🏋️

We will run the production-grade training loop, which includes: 
- Inverse-frequency weighted loss
- Mixed-precision training (AMP) to save GPU memory and double speed
- Cosine annealing learning rate scheduling
- Early stopping to prevent overfitting

In [ ]:
# @title Start GPU-Accelerated Training 🚀
# Adjust training arguments:
#   --num-classes: 3 (default: pupil/limbus/background), set to 4 if training with suction ring labels.
#   --epochs: set to 300 to achieve production accuracy.
#   --batch-size: 16 (GPU can easily handle 16 or 32 for ResNet34).
#   --device: cuda (strictly forces training on the active GPU).

!python scripts/train_model.py \
    --epochs 300 \
    --batch-size 16 \
    --lr 0.0005 \
    --num-classes 3 \
    --annotation-path clinical_data/annotations/annotations.json \
    --image-dir clinical_data/training_data/images \
    --mask-dir clinical_data/training_data/masks \
    --save-dir models \
    --device cuda

## 5. Export to Optimized ONNX & Download 📤

Convert your newly trained PyTorch model (`.pth`) to an optimized ONNX model (`.onnx`).  
ONNX models are highly recommended for the local tracking application because they require **no PyTorch dependency, run in a 50MB runtime, and execute much faster on CPUs**.

In [ ]:
# @title Export Checkpoint to ONNX 🌟
!python scripts/export_onnx.py --model models/best_model.pth --output models/onnx/segmentation.onnx

In [ ]:
# @title Package and Download Weights 💾
# Compresses model weights (both .pth and .onnx) into a single zip file for download.
import shutil
from google.colab import files
from pathlib import Path

models_dir = Path("/content/workspace/models")
archive_name = "/content/trained_models_colab"

if models_dir.exists():
    shutil.make_archive(archive_name, 'zip', str(models_dir))
    print("✅ Models packaged into '/content/trained_models_colab.zip'!")
    
    # Initiate download
    try:
        files.download('/content/trained_models_colab.zip')
        print("Download prompt started! Extract contents into your local project 'models/' folder.")
    except Exception as e:
        print(f"\n⚠️ Auto-download blocked/failed: {e}")
        print("Please manually download '/content/trained_models_colab.zip' from the left sidebar file manager.")
else:
    print("❌ Models folder not found! Ensure training completed successfully.")